# Chest X-ray Intelligence Studio — Unified Multi-Model Colab Training

**Project Title:** A Classifier with Visual Explanations for Predictions on Image Data Using Graph-Based Retrieval-Augmented Generation Systems.

This Google Colab notebook provides the self-healing, automated training and evaluation pipeline for all four architectures:
1. **Vision Transformer (ViT-B/16)** — Primary Model (Hugging Face `google/vit-base-patch16-224`)
2. **DenseNet121** — CNN Baseline (Feature reuse across dense blocks)
3. **Capsule Network (CapsNet)** — Part-whole modeling with dynamic routing
4. **Graph Attention Network (GAT)** — Spatial image-region graph reasoning (49 nodes, 8-neighbor edges)

Dataset: **CheXpert-v1.0-small** downloaded automatically via KaggleHub.

In [ ]:
# Cell 1: Hardware Environment Check
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.")

In [ ]:
# Cell 2: Install Required Libraries
!pip install -q kagglehub transformers accelerate scikit-learn networkx tqdm opencv-python-headless pillow pandas numpy matplotlib

In [ ]:
# Cell 3: Self-Healing Project Discovery
# Solves the folder mismatch issue: dynamically discovers the project root regardless of zip structure
import os, sys, zipfile
from pathlib import Path

def locate_project_root(search_dir="/content"):
    search_path = Path(search_dir)
    # Check all subdirectories for backend/train.py
    for candidate in [search_path] + list(search_path.glob("**/*")):
        if candidate.is_dir() and (candidate / "backend" / "train.py").is_file():
            return candidate
    return None

project_root = locate_project_root()

if project_root is None:
    print("Project source not found in /content. Please upload project ZIP:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.lower().endswith(".zip"):
            print(f"Extracting {fname}...")
            with zipfile.ZipFile(fname, "r") as zip_ref:
                zip_ref.extractall("/content")
    project_root = locate_project_root()

if project_root is None:
    raise FileNotFoundError("Could not locate 'backend/train.py' in extracted files. Please verify ZIP structure.")

print("✓ Project Root Discovered at:", project_root)
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
# Cell 4: Download CheXpert-v1.0-small via KaggleHub
import kagglehub
from backend.data import find_dataset_root

print("Downloading CheXpert dataset via KaggleHub...")
download_dir = kagglehub.dataset_download("ashery/chexpert")
data_root = str(find_dataset_root(download_dir))
print("✓ Verified CheXpert Root:", data_root)


## 5. Automated Multi-Model Training

Train models one by one or all together. Set `SMOKE_TEST = True` for a quick 500-sample verification run, or `False` for the full training run.

In [ ]:
SMOKE_TEST = False  # Set to True for a quick pipeline verification
MAX_TRAIN = 2000 if SMOKE_TEST else 20000
MAX_VALID = 200 if SMOKE_TEST else 0  # 0 = evaluate all valid.csv samples
EPOCHS = 2 if SMOKE_TEST else 5
UNCERTAIN_POLICY = "zero"  # Choices: 'zero', 'ones', 'masked'

print(f"Training Settings -> Epochs: {EPOCHS} | Train Samples: {MAX_TRAIN} | Valid Samples: {MAX_VALID} | Policy: {UNCERTAIN_POLICY}")

In [ ]:
# Train Model 1: Vision Transformer (ViT-B/16) [PRIMARY MODEL]
!python -m backend.train --model vit --data-root "{data_root}" --epochs {EPOCHS} --batch-size 16 --learning-rate 2e-5 --max-train-samples {MAX_TRAIN} --max-valid-samples {MAX_VALID} --uncertain-policy {UNCERTAIN_POLICY}

In [ ]:
# Train Model 2: DenseNet121 [CNN Baseline]
!python -m backend.train --model densenet --data-root "{data_root}" --epochs {EPOCHS} --batch-size 32 --learning-rate 1e-4 --max-train-samples {MAX_TRAIN} --max-valid-samples {MAX_VALID} --uncertain-policy {UNCERTAIN_POLICY}

In [ ]:
# Train Model 3: Capsule Network (CapsNet) [Dynamic Routing]
!python -m backend.train --model capsnet --data-root "{data_root}" --epochs {EPOCHS} --batch-size 16 --learning-rate 1e-4 --max-train-samples {MAX_TRAIN} --max-valid-samples {MAX_VALID} --uncertain-policy {UNCERTAIN_POLICY}

In [ ]:
# Train Model 4: Graph Attention Network (GAT) [Region Graph]
!python -m backend.train --model gat --data-root "{data_root}" --epochs {EPOCHS} --batch-size 16 --learning-rate 1e-4 --max-train-samples {MAX_TRAIN} --max-valid-samples {MAX_VALID} --uncertain-policy {UNCERTAIN_POLICY}

In [ ]:
# Export Trained Checkpoints to a downloadable ZIP for your Windows Laptop
!zip -r /content/trained_checkpoints.zip checkpoints/
from google.colab import files
print("Downloading trained_checkpoints.zip...")
files.download("/content/trained_checkpoints.zip")